# 1. Inicialización del Entorno Spark

Unidad Académica: **Demanda histórica y predicción temporal**

Integrante: **Harry Jack Ascuna Mamani** — Data Geniuses

Inicializamos la sesión **Spark** con una memoria de driver acotada (2g) para procesar
los ~5 millones de registros de Citi Bike NYC de forma distribuida.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("U1_Demanda_Historica_CitiBike")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 17:42:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


# 2. Definición del Esquema Explícito y Carga de Datos

Definimos un **esquema explícito** con `StructType` para tipar correctamente cada
columna del dataset de Citi Bike y evitar inferencias erróneas al leer los CSV.


In [2]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, DoubleType,
)

schema_citibike = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

DATA_PATH = "/opt/UNIDAD1/data/*.csv"

df = spark.read.schema(schema_citibike).option("header", True).csv(DATA_PATH)

26/09/11 17:42:04 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/UNIDAD1/data/*.csv.
java.io.FileNotFoundException: File /opt/UNIDAD1/data/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis

# 2.1 Verificación Inicial de Registros

Validamos la ingesta: número total de viajes cargados y estructura del DataFrame.


In [3]:
total_registros = df.count()
print(f"Registros ingeridos desde {DATA_PATH}: {total_registros:,}")

Registros ingeridos desde /opt/UNIDAD1/data/*.csv: 4,993,137


In [4]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



In [5]:
df.show(5, vertical=True, truncate=22)

-RECORD 0------------------------------------
 ride_id            | FB2708B4615FC7B1       
 rideable_type      | electric_bike          
 started_at         | 2026-07-04 15:52:29... 
 ended_at           | 2026-07-04 16:03:08... 
 start_station_name | W 17 St & 7 Ave        
 start_station_id   | 6107.08                
 end_station_name   | Grand St & Greene St   
 end_station_id     | 5500.02                
 start_lat          | 40.74056423633952      
 start_lng          | -73.99852573871613     
 end_lat            | 40.72170005235607      
 end_lng            | -74.00238141417502     
 member_casual      | member                 
-RECORD 1------------------------------------
 ride_id            | 6D9C1D2C900149A3       
 rideable_type      | classic_bike           
 started_at         | 2026-07-09 00:31:21... 
 ended_at           | 2026-07-09 00:35:10... 
 start_station_name | Cleveland Pl & Spri... 
 start_station_id   | 5492.05                
 end_station_name   | Grand St & G

# 3. Transformaciones y Limpieza de Calidad (Capa Silver)

Calculamos la **duración de cada viaje** en minutos y filtramos los registros válidos:
duración entre **1.0 y 180.0 minutos** y sin nulos en los campos clave para el modelado.


In [6]:
from pyspark.sql.functions import col, unix_timestamp

df = df.withColumn(
    "duration_minutes",
    (unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60.0,
)

In [7]:
df_clean = (
    df.filter(col("duration_minutes").between(1.0, 180.0))
      .filter(
          col("start_station_name").isNotNull()
          & col("end_station_name").isNotNull()
          & col("rideable_type").isNotNull()
          & col("member_casual").isNotNull()
      )
)

n_validos = df_clean.count()
print(f"Viajes válidos tras limpieza Silver: {n_validos:,}")
print(f"Registros descartados: {total_registros - n_validos:,}")

Viajes válidos tras limpieza Silver: 4,972,819
Registros descartados: 20,318


# 3.1 Inspección de Muestra Limpia

Mostramos una muestra de los viajes limpios con las columnas esenciales.


In [8]:
df_clean.select(
    "started_at", "ended_at", "duration_minutes",
    "rideable_type", "start_station_name", "member_casual",
).show(5)

+--------------------+--------------------+------------------+-------------+--------------------+-------------+
|          started_at|            ended_at|  duration_minutes|rideable_type|  start_station_name|member_casual|
+--------------------+--------------------+------------------+-------------+--------------------+-------------+
|2026-07-04 15:52:...|2026-07-04 16:03:...|             10.65|electric_bike|     W 17 St & 7 Ave|       member|
|2026-07-09 00:31:...|2026-07-09 00:35:...| 3.816666666666667| classic_bike|Cleveland Pl & Sp...|       member|
|2026-07-02 22:51:...|2026-07-02 22:56:...| 4.716666666666667|electric_bike|Cathedral Pkwy & ...|       member|
|2026-07-01 15:50:...|2026-07-01 15:58:...| 7.383333333333334|electric_bike|Cleveland Pl & Sp...|       member|
|2026-07-08 06:13:...|2026-07-08 06:33:...|19.816666666666666| classic_bike|     6 Ave & W 34 St|       member|
+--------------------+--------------------+------------------+-------------+--------------------+-------

# 4. Feature Engineering: Extracción de Factores Temporales

A partir de `started_at` extraemos las variables explicativas temporales:
- `trip_date` (fecha del viaje)
- `start_hour` (hora 0-23)
- `day_of_week` (día de la semana 1-7, ISO)
- `is_weekend` (binario: 1 = fin de semana, 0 = entre semana)


In [9]:
from pyspark.sql.functions import (
    col, to_date, hour, dayofweek,
    when, year, month, min, max,
)

df_feat = (
    df_clean.withColumn("trip_date", to_date(col("started_at")))
            .withColumn("start_hour", hour(col("started_at")))
            .withColumn("day_of_week", dayofweek(col("started_at")))
            .withColumn(
                "is_weekend",
                when(col("day_of_week").isin(1, 7), 1).otherwise(0),
            )
            .withColumn("trip_year", year(col("started_at")))
            .withColumn("trip_month", month(col("started_at")))
)

In [10]:
print("Distribución temporal de viajes:")
df_feat.groupBy("trip_date").count().orderBy("trip_date").show(5, truncate=False)
print("Rango de fechas:", df_feat.agg(min("trip_date"), max("trip_date")).collect()[0])

Distribución temporal de viajes:


+----------+------+
|trip_date |count |
+----------+------+
|2026-06-30|777   |
|2026-07-01|179310|
|2026-07-02|148743|
|2026-07-03|109311|
|2026-07-04|115800|
+----------+------+
only showing top 5 rows


Rango de fechas: Row(min(trip_date)=datetime.date(2026, 6, 30), max(trip_date)=datetime.date(2026, 7, 31))


# 5. Construcción del Dataset de Demanda Agregada (Variable Objetivo)

Agregamos la demanda **(Y)** por combinación de factores temporales y demográficos.
Cada fila representa un conjunto de features (`X`) con su demanda de viajes (`trip_count`).


In [11]:
from pyspark.sql.functions import count

df_demanda = (
    df_feat.groupBy(
        "trip_date", "start_hour", "day_of_week", "is_weekend",
        "member_casual", "rideable_type",
    )
    .agg(count("*").alias("trip_count"))
)

print("Particiones del dataset agregado:", df_demanda.rdd.getNumPartitions())

Particiones del dataset agregado: 1


# 5.1 Comprobación del Dataset de Demanda

Confirmamos la estructura de features del dataset agregado: muestra y total de filas.


In [12]:
print(f"Total de filas del dataset agregado: {df_demanda.count():,}")
df_demanda.orderBy("trip_date", "start_hour").show(10, truncate=False)

Total de filas del dataset agregado: 2,983


+----------+----------+-----------+----------+-------------+-------------+----------+
|trip_date |start_hour|day_of_week|is_weekend|member_casual|rideable_type|trip_count|
+----------+----------+-----------+----------+-------------+-------------+----------+
|2026-06-30|21        |3          |0         |member       |electric_bike|2         |
|2026-06-30|22        |3          |0         |member       |electric_bike|5         |
|2026-06-30|22        |3          |0         |casual       |electric_bike|14        |
|2026-06-30|22        |3          |0         |member       |classic_bike |1         |
|2026-06-30|23        |3          |0         |member       |electric_bike|405       |
|2026-06-30|23        |3          |0         |member       |classic_bike |100       |
|2026-06-30|23        |3          |0         |casual       |classic_bike |59        |
|2026-06-30|23        |3          |0         |casual       |electric_bike|191       |
|2026-07-01|0         |4          |0         |member  

In [13]:
print("Estadística descriptiva de la demanda (trip_count):")
df_demanda.describe("trip_count").show()

Estadística descriptiva de la demanda (trip_count):


+-------+------------------+
|summary|        trip_count|
+-------+------------------+
|  count|              2983|
|   mean|1667.0529668119343|
| stddev|2016.3839127786887|
|    min|                 1|
|    max|             12186|
+-------+------------------+



# 6. Pipeline de Machine Learning (Spark MLlib)

Preparamos las variables explicativas para el modelo:
- `StringIndexer` codifica las categóricas (`member_casual`, `rideable_type`) a numérico.
- `OneHotEncoder` convierte los índices en vectores sparse.
- `VectorAssembler` empaqueta temporal + categóricas codificadas en la columna `features`.


In [14]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

categoricas = ["member_casual", "rideable_type"]
numericas = ["start_hour", "day_of_week", "is_weekend"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx").setHandleInvalid("keep")
    for c in categoricas
]
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categoricas],
    outputCols=[c + "_ohe" for c in categoricas],
)
assembler = VectorAssembler(
    inputCols=numericas + [c + "_ohe" for c in categoricas],
    outputCol="features",
)

# 7. División de Datos y Entrenamiento del Modelo de Regresión

Dividimos en **Train (80%)** y **Test (20%)** con semilla reproducible (42) y
entrenamos un regresor distribuido de Spark MLlib con `labelCol="trip_count"`.


In [15]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + [encoder, assembler])
df_model = pipeline.fit(df_demanda).transform(df_demanda)

train, test = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,} filas | Test: {test.count():,} filas")

Train: 2,439 filas | Test: 544 filas


In [16]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="trip_count")
model = lr.fit(train)
print("Modelo entrenado correctamente.")

26/09/11 17:42:33 WARN Instrumentation: [d91d9fee] regParam is zero, which might cause numerical instability and overfitting.


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
26/09/11 17:42:34 WARN Instrumentation: [d91d9fee] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


Modelo entrenado correctamente.


In [17]:
print("Coeficientes del modelo:", model.coefficients)
print("Intercepto:", model.intercept)

Coeficientes del modelo: [92.12289589637396,6.160673586194607,-164.29744889730165,993.330622345448,-993.3306223454482,775.1340099261129,-775.1340099261097]
Intercepto: 621.8806844617698


# 8. Inferencia y Predicción sobre Conjunto de Prueba

Aplicamos el modelo entrenado sobre el conjunto de **test** y comparamos
`trip_count` real frente a `prediction` estimada.


In [18]:
predicciones = model.transform(test)
predicciones.select("trip_count", "prediction", "features").show(10, truncate=False)

+----------+-------------------+------------------------------+
|trip_count|prediction         |features                      |
+----------+-------------------+------------------------------+
|1         |2885.263027359919  |[22.0,3.0,0.0,1.0,0.0,0.0,1.0]|
|100       |2977.385923256293  |[23.0,3.0,0.0,1.0,0.0,0.0,1.0]|
|167       |-1121.9412534650096|(7,[1,4,6],[4.0,1.0,1.0])     |
|335       |520.4496622835868  |[1.0,4.0,0.0,0.0,1.0,1.0,0.0] |
|407       |2599.233802870857  |[2.0,4.0,0.0,1.0,0.0,1.0,0.0] |
|241       |2691.3566987672307 |[3.0,4.0,0.0,1.0,0.0,1.0,0.0] |
|183       |888.9412458690826  |[5.0,4.0,0.0,0.0,1.0,1.0,0.0] |
|2659      |2967.7253864563527 |[6.0,4.0,0.0,1.0,0.0,1.0,0.0] |
|961       |1257.4328294545785 |[9.0,4.0,0.0,0.0,1.0,1.0,0.0] |
|2776      |1693.8260542932521 |[9.0,4.0,0.0,1.0,0.0,0.0,1.0] |
+----------+-------------------+------------------------------+
only showing top 10 rows


# 9. Evaluación Formal del Modelo (Métricas de Calidad)

Calculamos métricas estándar de regresión con `RegressionEvaluator`:
- **RMSE** (Root Mean Squared Error)
- **MAE** (Mean Absolute Error)
- **R2** (Coeficiente de determinación)


In [19]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="trip_count", predictionCol="prediction", metricName="r2")

rmse = rmse_eval.evaluate(predicciones)
mae = mae_eval.evaluate(predicciones)
r2 = r2_eval.evaluate(predicciones)

In [20]:
print("=" * 52)
print("MÉTRICAS DE CALIDAD DEL MODELO (Capa Silver/Gold)")
print("=" * 52)
print(f"  RMSE : {rmse:10.4f}  (Root Mean Squared Error)")
print(f"  MAE  : {mae:10.4f}  (Mean Absolute Error)")
print(f"  R2   : {r2:10.4f}  (Coeficiente de determinación)")
print("=" * 52)

MÉTRICAS DE CALIDAD DEL MODELO (Capa Silver/Gold)
  RMSE :  1439.6690  (Root Mean Squared Error)
  MAE  :  1028.4467  (Mean Absolute Error)
  R2   :     0.4805  (Coeficiente de determinación)


# 10. Persistencia Analítica de Predicciones (Capa Gold)

Persistimos el resultado final en formato **Parquet** (columnar y comprimido) en la
carpeta analítica Gold para su posterior consumo por los reportes del dashboard.


In [21]:
from pyspark.sql.functions import round as spark_round, col as ccol

pred_final = (
    predicciones.select(
        ccol("trip_date"),
        ccol("start_hour"),
        ccol("day_of_week"),
        ccol("is_weekend"),
        ccol("member_casual"),
        ccol("rideable_type"),
        ccol("trip_count"),
        spark_round(ccol("prediction"), 2).alias("prediction"),
    )
)

GOLD_PATH = "/opt/UNIDAD1/gold/demanda_historica_prediccion"
pred_final.write.mode("overwrite").parquet(GOLD_PATH)
print(f"Predicciones persistidas en Parquet: {GOLD_PATH}")

Predicciones persistidas en Parquet: /opt/UNIDAD1/gold/demanda_historica_prediccion


In [22]:
gold_check = spark.read.parquet(GOLD_PATH)
print(f"Filas en capa Gold: {gold_check.count():,}")
gold_check.show(5, truncate=False)

Filas en capa Gold: 544
+----------+----------+-----------+----------+-------------+-------------+----------+----------+
|trip_date |start_hour|day_of_week|is_weekend|member_casual|rideable_type|trip_count|prediction|
+----------+----------+-----------+----------+-------------+-------------+----------+----------+
|2026-06-30|22        |3          |0         |member       |classic_bike |1         |2885.26   |
|2026-06-30|23        |3          |0         |member       |classic_bike |100       |2977.39   |
|2026-07-01|0         |4          |0         |casual       |classic_bike |167       |-1121.94  |
|2026-07-01|1         |4          |0         |casual       |electric_bike|335       |520.45    |
|2026-07-01|2         |4          |0         |member       |electric_bike|407       |2599.23   |
+----------+----------+-----------+----------+-------------+-------------+----------+----------+
only showing top 5 rows
